# Library

In [1]:
import os
import gc
import copy
import random
import warnings
import ctypes

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    warnings.filterwarnings("ignore")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)

    def seed_worker(worker_id: int) -> None:
        worker_seed = torch.initial_seed() % (2 ** 32)
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    print(f"Random seed: {seed}")
    return seed_worker


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

# Data

In [3]:
DATASET_CONFIG = {
    "sunspots": {
        "path": "/kaggle/input/datasets/cryandrrich/time-series-prediction/sunspots_dataset.csv",
        "time_col": "Date",
        "feature_cols": ["Monthly Mean Total Sunspot Number"],
        "target_cols": ["Monthly Mean Total Sunspot Number"],
    },
    "appliances_energy": {
        "path": "/kaggle/input/datasets/cryandrrich/time-series-prediction/appliances_energy_dataset.csv",
        "time_col": "date",
        "feature_cols": ["Appliances", "lights"],
        "target_cols": [
            "Appliances", "lights", "T1", "RH_1", "T2", "RH_2", "T3", "RH_3",
            "T4", "RH_4", "T5", "RH_5", "T6", "RH_6", "T7", "RH_7",
            "T8", "RH_8", "T9", "RH_9",
        ],
    },
    "beijing_air_quality": {
        "path": "/kaggle/input/datasets/cryandrrich/time-series-prediction/beijing_air_quality_dataset.csv",
        "time_col": ["year", "month", "day", "hour"],
        "feature_cols": [
            "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
            "TEMP", "PRES", "DEWP", "RAIN", "WSPM",
        ],
        "target_cols": ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"],
    },
    "hanoi_air_quality": {
        "path": "/kaggle/input/datasets/cryandrrich/time-series-prediction/hanoi_air_quality_dataset.csv",
        "time_col": "Local Time",
        "feature_cols": ["AQI", "CO", "NO2", "O3", "PM10", "PM25", "SO2"],
        "target_cols": [
            "AQI", "CO", "NO2", "O3", "PM10", "PM25", "SO2",
            "Clouds", "Precipitation", "Pressure", "Relative Humidity",
            "Temperature", "UV Index", "Wind Speed",
        ],
    },
    "bitcoin": {
        "path": "/kaggle/input/datasets/cryandrrich/time-series-prediction/bitcoin_dataset.csv",
        "time_col": "Timestamp",
        "feature_cols": ["Open"],
        "target_cols": ["Open"],
    }
}

In [4]:
class TSWindowDataset(Dataset):
    def __init__(self, feature_df, target_df, seq_len, pred_len):
        feature_df = feature_df.apply(pd.to_numeric, errors="coerce").astype(np.float32)
        target_df = target_df.apply(pd.to_numeric, errors="coerce").astype(np.float32)
        feature_df = feature_df.ffill().bfill()
        target_df = target_df.ffill().bfill()

        self.X_data = torch.tensor(feature_df.to_numpy(dtype=np.float32), dtype=torch.float32)
        self.y_data = torch.tensor(target_df.to_numpy(dtype=np.float32), dtype=torch.float32)
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.X_data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        X = self.X_data[idx : idx + self.seq_len]
        y = self.y_data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return X, y


class TimeSeriesDataset:
    def __init__(
        self,
        name,
        seq_len=24,
        pred_len=1,
        train_ratio=0.7,
        val_ratio=0.2,
        test_ratio=0.1,
        batch_size=64,
        num_workers=0,
        worker_init_fn=None,
        generator=None,
        normalize=True,
    ):
        if name not in DATASET_CONFIG:
            raise ValueError(f"Dataset '{name}' không có trong DATASET_CONFIG.")
        if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
            raise ValueError("train_ratio + val_ratio + test_ratio phải = 1.0")

        self.name = name
        self.config = DATASET_CONFIG[name].copy()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        self.test_ratio = test_ratio
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.worker_init_fn = worker_init_fn
        self.generator = generator
        self.normalize = normalize

        self.scaler_stats = {}
        self.feature_cols = None
        self.target_cols = None

        self._load_and_preprocess()
        self._split_data()
        if self.normalize:
            self._normalize_data()
        self._create_datasets()

    # -------- LOAD --------
    def _load_and_preprocess(self):
        df = pd.read_csv(self.config["path"])
        time_col = self.config.get("time_col")
        if time_col is None:
            raise ValueError("Thiếu 'time_col' trong DATASET_CONFIG.")

        if isinstance(time_col, list):
            df["datetime"] = pd.to_datetime(df[time_col], errors="coerce")
        else:
            df["datetime"] = pd.to_datetime(df[time_col], errors="coerce")

        df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

        for col in ["No", "station"]:
            if col in df.columns:
                df = df.drop(columns=[col])

        # Cyclical time features
        df["hour"] = df["datetime"].dt.hour
        df["month"] = df["datetime"].dt.month
        df["dayofweek"] = df["datetime"].dt.dayofweek
        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
        df["dayofweek_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
        df["dayofweek_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

        # Wind direction
        if "wd" in df.columns:
            if df["wd"].isna().sum() > 0:
                df["wd"] = df["wd"].fillna(df["wd"].mode()[0])
            df = pd.get_dummies(df, columns=["wd"], prefix="wd", dtype=np.float32)

        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        df[numeric_cols] = df[numeric_cols].ffill().bfill()

        target_cols = self.config.get("target_cols")
        feature_cols = self.config.get("feature_cols")
        if target_cols is None:
            raise ValueError("Thiếu 'target_cols' trong DATASET_CONFIG.")
        if isinstance(target_cols, str):
            target_cols = [target_cols]

        if feature_cols is None:
            feature_cols = [
                c for c in df.select_dtypes(include=[np.number, bool]).columns
                if c != "datetime"
            ]
        else:
            feature_cols = list(feature_cols)

        for c in [c for c in df.columns if c.startswith("wd_")]:
            if c not in feature_cols:
                feature_cols.append(c)

        missing_f = [c for c in feature_cols if c not in df.columns]
        missing_t = [c for c in target_cols if c not in df.columns]
        if missing_f:
            raise ValueError(f"Feature_cols thiếu: {missing_f}")
        if missing_t:
            raise ValueError(f"Target_cols thiếu: {missing_t}")

        keep_cols = ["datetime"] + list(dict.fromkeys(feature_cols + target_cols))
        df = df[keep_cols]

        self.feature_cols = feature_cols
        self.target_cols = target_cols
        self.df_processed = df

    # -------- SPLIT --------
    def _split_data(self):
        df = self.df_processed.copy()
        n = len(df)
        train_end = int(n * self.train_ratio)
        val_end = int(n * (self.train_ratio + self.val_ratio))

        self.df_train = df.iloc[:train_end].reset_index(drop=True)
        self.df_val = df.iloc[train_end:val_end].reset_index(drop=True)
        self.df_test = df.iloc[val_end:].reset_index(drop=True)

        self.X_train = self.df_train[self.feature_cols].copy()
        self.X_val = self.df_val[self.feature_cols].copy()
        self.X_test = self.df_test[self.feature_cols].copy()
        self.y_train = self.df_train[self.target_cols].copy()
        self.y_val = self.df_val[self.target_cols].copy()
        self.y_test = self.df_test[self.target_cols].copy()

        for attr in ["X_train", "X_val", "X_test", "y_train", "y_val", "y_test"]:
            d = getattr(self, attr)
            d = d.apply(pd.to_numeric, errors="coerce").astype(np.float32)
            d = d.ffill().bfill()
            setattr(self, attr, d)

    # -------- NORMALIZE --------
    def _normalize_data(self):
        all_cols = list(dict.fromkeys(self.feature_cols + self.target_cols))
        train_full = self.df_train[all_cols]

        for col in all_cols:
            mean = float(train_full[col].mean())
            std = float(train_full[col].std())
            if std == 0 or np.isnan(std):
                std = 1.0
            self.scaler_stats[col] = {"mean": mean, "std": std}

        def apply_scale(df_part):
            df_scaled = df_part.copy()
            for col in df_scaled.columns:
                m = self.scaler_stats[col]["mean"]
                s = self.scaler_stats[col]["std"]
                df_scaled[col] = (df_scaled[col] - m) / s
            return df_scaled.astype(np.float32)

        self.X_train = apply_scale(self.X_train)
        self.X_val = apply_scale(self.X_val)
        self.X_test = apply_scale(self.X_test)
        self.y_train = apply_scale(self.y_train)
        self.y_val = apply_scale(self.y_val)
        self.y_test = apply_scale(self.y_test)

    def inverse_transform_target(self, y_scaled):
        """Đưa y (đã scale) về giá trị thật. Shape (..., len(target_cols))."""
        if not self.normalize or not self.scaler_stats:
            return y_scaled
        if isinstance(y_scaled, torch.Tensor):
            y_scaled = y_scaled.detach().cpu().numpy()

        original_shape = y_scaled.shape
        y_2d = y_scaled.reshape(-1, len(self.target_cols))
        means = np.array([self.scaler_stats[c]["mean"] for c in self.target_cols], dtype=np.float32)
        stds = np.array([self.scaler_stats[c]["std"] for c in self.target_cols], dtype=np.float32)
        return (y_2d * stds + means).reshape(original_shape)

    # -------- DATASETS / LOADERS --------
    def _create_datasets(self):
        self.train_dataset = TSWindowDataset(self.X_train, self.y_train, self.seq_len, self.pred_len)
        self.val_dataset = TSWindowDataset(self.X_val, self.y_val, self.seq_len, self.pred_len)
        self.test_dataset = TSWindowDataset(self.X_test, self.y_test, self.seq_len, self.pred_len)

    def get_loaders(self):
        def make(ds, shuffle):
            return DataLoader(
                ds,
                batch_size=self.batch_size,
                shuffle=shuffle,
                num_workers=self.num_workers,
                worker_init_fn=self.worker_init_fn,
                generator=self.generator,
            )
        return make(self.train_dataset, True), make(self.val_dataset, False), make(self.test_dataset, False)

    # Tiện ích: lấy index của target trong feature_cols (cho residual model, baseline)
    def get_target_indices_in_features(self):
        return [self.feature_cols.index(c) for c in self.target_cols if c in self.feature_cols]

    def print_info(self):
        X, y = self.train_dataset[0]
        print(f"=== DATASET: {self.name.upper()} ===")
        print(f"- Tổng số dòng: {len(self.df_processed)}")
        print(f"- seq_len / pred_len: {self.seq_len} / {self.pred_len} | batch_size: {self.batch_size}")
        print(f"- Normalize: {self.normalize}")
        print(f"- #features: {len(self.feature_cols)} | #targets: {len(self.target_cols)}")
        print(f"- X shape: {tuple(X.shape)} | y shape: {tuple(y.shape)}")
        print(
            f"- Split (train/val/test): "
            f"{len(self.df_train)}/{len(self.df_val)}/{len(self.df_test)} dòng | "
            f"{len(self.train_dataset)}/{len(self.val_dataset)}/{len(self.test_dataset)} cửa sổ"
        )
        print("=" * 60)

# Evaluate

In [5]:
def wmape(y_true, y_pred, eps=1e-8):
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + eps)


def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return {
        "MSE": mse,
        "MAE": mae,
        "RMSE": np.sqrt(mse),
        "WMAPE": wmape(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


def _format_metrics(m):
    return (
        f"MSE: {m['MSE']:.4f} | MAE: {m['MAE']:.4f} | "
        f"RMSE: {m['RMSE']:.4f} | WMAPE: {m['WMAPE']:.4f} | R2: {m['R2']:.4f}"
    )

# Train

In [6]:
class Trainer:
    """
    Trainer độc lập với model. Chỉ cần model nhận input shape (B, seq_len, n_features)
    và output shape (B, pred_len, target_dim).
    """

    def __init__(
        self,
        model,
        dataset: TimeSeriesDataset,
        train_loader,
        val_loader,
        test_loader,
        criterion=None,
        optimizer=None,
        scheduler=None,
        device="cuda",
        grad_clip=1.0,
    ):
        self.model = model.to(device)
        self.dataset = dataset
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader

        self.criterion = criterion if criterion is not None else nn.MSELoss()
        self.optimizer = optimizer if optimizer is not None else torch.optim.AdamW(
            self.model.parameters(), lr=1e-3, weight_decay=1e-4
        )
        self.scheduler = scheduler
        self.device = device
        self.grad_clip = grad_clip

        self.best_model_state = None
        self.history = {"train_loss": [], "val_loss": []}

    # ---- core loops ----
    def _train_one_epoch(self):
        self.model.train()
        total_loss, total_n = 0.0, 0
        for X, y in self.train_loader:
            X = X.to(self.device)
            y = y.to(self.device)

            self.optimizer.zero_grad()
            y_pred = self.model(X)

            if y_pred.shape != y.shape:
                raise ValueError(f"Shape mismatch: y_pred={y_pred.shape}, y={y.shape}")

            loss = self.criterion(y_pred, y)
            loss.backward()
            if self.grad_clip:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            self.optimizer.step()

            bs = X.size(0)
            total_loss += loss.item() * bs
            total_n += bs
        return total_loss / total_n

    def _eval_loss(self, loader):
        self.model.eval()
        total_loss, total_n = 0.0, 0
        with torch.no_grad():
            for X, y in loader:
                X = X.to(self.device)
                y = y.to(self.device)
                loss = self.criterion(self.model(X), y)
                bs = X.size(0)
                total_loss += loss.item() * bs
                total_n += bs
        return total_loss / total_n

    @torch.no_grad()
    def _predict(self, loader):
        """Return (y_true, y_pred) ở giá trị thật (đã inverse scale nếu có)."""
        self.model.eval()
        preds, trues = [], []
        for X, y in loader:
            X = X.to(self.device)
            y_pred = self.model(X)
            preds.append(y_pred.detach().cpu())
            trues.append(y.detach().cpu())

        y_pred = torch.cat(preds, dim=0).numpy()
        y_true = torch.cat(trues, dim=0).numpy()

        if self.dataset.normalize:
            y_pred = self.dataset.inverse_transform_target(y_pred)
            y_true = self.dataset.inverse_transform_target(y_true)
        return y_true, y_pred

    # ---- public API ----
    def fit(self, num_epochs=80, patience=15, verbose=True, eval_every=1):
        best_val = float("inf")
        counter = 0

        for epoch in range(1, num_epochs + 1):
            train_loss = self._train_one_epoch()
            val_loss = self._eval_loss(self.val_loader)

            if self.scheduler is not None:
                if isinstance(self.scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    self.scheduler.step(val_loss)
                else:
                    self.scheduler.step()

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)

            if verbose:
                lr = self.optimizer.param_groups[0]["lr"]
                msg = (
                    f"Epoch [{epoch:03d}/{num_epochs}] "
                    f"Train: {train_loss:.6f} | Val: {val_loss:.6f} | lr: {lr:.2e}"
                )
                if eval_every and epoch % eval_every == 0:
                    val_overall = self.evaluate(self.val_loader, verbose=False)["overall"]
                    msg += f" | Val MAE: {val_overall['MAE']:.4f}"
                print(msg)

            if val_loss < best_val:
                best_val = val_loss
                self.best_model_state = copy.deepcopy(self.model.state_dict())
                counter = 0
            else:
                counter += 1
                if counter >= patience:
                    print(f"Early stopping ở epoch {epoch}.")
                    break

        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
        print("Train xong.")
        return self.history

    def evaluate(self, loader=None, verbose=True, title="EVALUATION"):
        """
        Trả về dict {'overall': metrics, 'per_target': {col: metrics}}.
        Univariate (1 target) chỉ in overall.
        """
        if loader is None:
            loader = self.test_loader

        y_true, y_pred = self._predict(loader)
        target_dim = len(self.dataset.target_cols)

        y_true_2d = y_true.reshape(-1, target_dim)
        y_pred_2d = y_pred.reshape(-1, target_dim)

        per_target = {}
        for i, col in enumerate(self.dataset.target_cols):
            per_target[col] = compute_metrics(y_true_2d[:, i], y_pred_2d[:, i])

        overall = compute_metrics(y_true_2d.reshape(-1), y_pred_2d.reshape(-1))

        if verbose:
            print(f"=== {title} ===")
            if target_dim > 1:
                for col, m in per_target.items():
                    print(f"{col:>10s} | {_format_metrics(m)}")
                print("-" * 80)
            print(f"OVERALL    | {_format_metrics(overall)}")
            print("=" * 80)

        return {"overall": overall, "per_target": per_target}

In [7]:
def evaluate_naive_baseline(dataset: TimeSeriesDataset, loader, verbose=True):
    """
    Baseline: y(t+k) = giá trị gần nhất của chính target đó trong input window.
    Yêu cầu mỗi target có mặt trong feature_cols.
    """
    target_indices = []
    for col in dataset.target_cols:
        if col not in dataset.feature_cols:
            raise ValueError(f"Target '{col}' không có trong feature_cols, không chạy naive baseline được.")
        target_indices.append(dataset.feature_cols.index(col))

    preds, trues = [], []
    for X, y in loader:
        y_pred = X[:, -1:, target_indices]            # (B, 1, target_dim)
        if y.shape[1] > 1:
            y_pred = y_pred.repeat(1, y.shape[1], 1)  # broadcast theo pred_len
        preds.append(y_pred.cpu())
        trues.append(y.cpu())

    y_pred = torch.cat(preds, dim=0).numpy()
    y_true = torch.cat(trues, dim=0).numpy()

    if dataset.normalize:
        y_pred = dataset.inverse_transform_target(y_pred)
        y_true = dataset.inverse_transform_target(y_true)

    target_dim = len(dataset.target_cols)
    y_true_2d = y_true.reshape(-1, target_dim)
    y_pred_2d = y_pred.reshape(-1, target_dim)

    per_target = {
        col: compute_metrics(y_true_2d[:, i], y_pred_2d[:, i])
        for i, col in enumerate(dataset.target_cols)
    }
    overall = compute_metrics(y_true_2d.reshape(-1), y_pred_2d.reshape(-1))

    if verbose:
        print("=== NAIVE BASELINE ===")
        if target_dim > 1:
            for col, m in per_target.items():
                print(f"{col:>10s} | {_format_metrics(m)}")
            print("-" * 80)
        print(f"OVERALL    | {_format_metrics(overall)}")
        print("=" * 80)

    return {"overall": overall, "per_target": per_target}


def compare_metrics(baseline_metrics, model_metrics, title="SO SÁNH OVERALL"):
    b, m = baseline_metrics["overall"], model_metrics["overall"]
    print(f"=== {title} ===")
    for k in ["MAE", "MSE", "RMSE", "WMAPE", "R2"]:
        print(f"{k:6s} | Baseline: {b[k]:.4f} | Model: {m[k]:.4f}")
    print("=" * 60)

# Model

In [8]:
# =============================================================================
# 1. SERIES DECOMPOSITION
# =============================================================================
class MovingAvg(nn.Module):
    """Moving average filter dùng để lấy trend."""
    def __init__(self, kernel_size, stride=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        # x: (B, L, C)
        pad = (self.kernel_size - 1) // 2
        front = x[:, :1, :].repeat(1, pad, 1)
        end = x[:, -1:, :].repeat(1, self.kernel_size - 1 - pad, 1)
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1)).permute(0, 2, 1)
        return x


class SeriesDecomp(nn.Module):
    """Tách x thành seasonal (residual) + trend (moving avg)."""
    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size, stride=1)

    def forward(self, x):
        trend = self.moving_avg(x)
        seasonal = x - trend
        return seasonal, trend


class DFTSeriesDecomp(nn.Module):
    """Decompose theo DFT: giữ top_k tần số làm seasonal."""
    def __init__(self, top_k=5):
        super().__init__()
        self.top_k = top_k

    def forward(self, x):
        # x: (B, L, C)
        xf = torch.fft.rfft(x, dim=1)
        amp = torch.abs(xf)
        # Lấy top_k tần số (bỏ DC)
        amp[:, 0, :] = 0
        _, top_idx = torch.topk(amp, self.top_k, dim=1)
        mask = torch.zeros_like(xf)
        mask.scatter_(1, top_idx, 1)
        x_season = torch.fft.irfft(xf * mask, n=x.size(1), dim=1)
        x_trend = x - x_season
        return x_season, x_trend


# =============================================================================
# 2. MULTI-SCALE MIXING
# =============================================================================
class MultiScaleSeasonMixing(nn.Module):
    """
    Bottom-up: fine → coarse.
    Mixing seasonal từ scale mịn nhất xuống các scale thô hơn.
    """
    def __init__(self, seq_len, down_sampling_window, down_sampling_layers):
        super().__init__()
        self.down_sampling_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(
                    seq_len // (down_sampling_window ** i),
                    seq_len // (down_sampling_window ** (i + 1)),
                ),
                nn.GELU(),
                nn.Linear(
                    seq_len // (down_sampling_window ** (i + 1)),
                    seq_len // (down_sampling_window ** (i + 1)),
                ),
            )
            for i in range(down_sampling_layers)
        ])

    def forward(self, season_list):
        # season_list: list các tensor (B, C, L_i) với L_i giảm dần
        out_high = season_list[0]
        out_low = season_list[1] if len(season_list) > 1 else None
        out_season_list = [out_high]

        for i in range(len(season_list) - 1):
            out_low_res = self.down_sampling_layers[i](out_high) + out_low
            out_high = out_low_res
            if i + 2 <= len(season_list) - 1:
                out_low = season_list[i + 2]
            out_season_list.append(out_high)
        return out_season_list


class MultiScaleTrendMixing(nn.Module):
    """
    Top-down: coarse → fine.
    Mixing trend từ scale thô nhất lên các scale mịn hơn.
    """
    def __init__(self, seq_len, down_sampling_window, down_sampling_layers):
        super().__init__()
        self.up_sampling_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(
                    seq_len // (down_sampling_window ** (i + 1)),
                    seq_len // (down_sampling_window ** i),
                ),
                nn.GELU(),
                nn.Linear(
                    seq_len // (down_sampling_window ** i),
                    seq_len // (down_sampling_window ** i),
                ),
            )
            for i in reversed(range(down_sampling_layers))
        ])

    def forward(self, trend_list):
        # Đảo thứ tự để bắt đầu từ coarse → fine
        trend_list_reverse = trend_list.copy()
        trend_list_reverse.reverse()

        out_low = trend_list_reverse[0]
        out_high = trend_list_reverse[1] if len(trend_list_reverse) > 1 else None
        out_trend_list = [out_low]

        for i in range(len(trend_list_reverse) - 1):
            out_high_res = self.up_sampling_layers[i](out_low) + out_high
            out_low = out_high_res
            if i + 2 <= len(trend_list_reverse) - 1:
                out_high = trend_list_reverse[i + 2]
            out_trend_list.append(out_low)

        out_trend_list.reverse()
        return out_trend_list


# =============================================================================
# 3. PAST DECOMPOSABLE MIXING (PDM) BLOCK
# =============================================================================
class PastDecomposableMixing(nn.Module):
    def __init__(
        self,
        seq_len,
        d_model,
        d_ff,
        dropout,
        down_sampling_window,
        down_sampling_layers,
        decomp_method="moving_avg",
        moving_avg=25,
        top_k=5,
        channel_independence=False,
    ):
        super().__init__()
        self.channel_independence = channel_independence

        if decomp_method == "moving_avg":
            self.decompsition = SeriesDecomp(moving_avg)
        elif decomp_method == "dft_decomp":
            self.decompsition = DFTSeriesDecomp(top_k)
        else:
            raise ValueError(f"decomp_method '{decomp_method}' không hợp lệ.")

        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        # Channel mixing (nếu không dùng channel independence)
        if not channel_independence:
            self.cross_layer = nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.GELU(),
                nn.Linear(d_ff, d_model),
            )

        self.mixing_multi_scale_season = MultiScaleSeasonMixing(
            seq_len, down_sampling_window, down_sampling_layers
        )
        self.mixing_multi_scale_trend = MultiScaleTrendMixing(
            seq_len, down_sampling_window, down_sampling_layers
        )

        self.out_cross_layer = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x_list):
        # x_list: list các tensor (B, L_i, d_model)
        length_list = [x.size(1) for x in x_list]

        season_list, trend_list = [], []
        for x in x_list:
            season, trend = self.decompsition(x)
            if not self.channel_independence:
                season = self.cross_layer(season)
                trend = self.cross_layer(trend)
            # (B, L_i, d_model) → (B, d_model, L_i) cho mixing theo trục thời gian
            season_list.append(season.permute(0, 2, 1))
            trend_list.append(trend.permute(0, 2, 1))

        # Multi-scale mixing
        out_season_list = self.mixing_multi_scale_season(season_list)
        out_trend_list = self.mixing_multi_scale_trend(trend_list)

        out_list = []
        for ori, out_season, out_trend, length in zip(
            x_list, out_season_list, out_trend_list, length_list
        ):
            # Permute lại: (B, d_model, L) → (B, L, d_model)
            out = out_season.permute(0, 2, 1) + out_trend.permute(0, 2, 1)
            out = ori + self.dropout(self.out_cross_layer(out))
            out_list.append(out[:, :length, :])
        return out_list


# =============================================================================
# 4. NORMALIZE LAYER (RevIN-like, simplified)
# =============================================================================
class Normalize(nn.Module):
    """Instance normalization theo timestep, có thể inverse."""
    def __init__(self, num_features, eps=1e-5, affine=True, non_norm=False):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        self.non_norm = non_norm
        if affine:
            self.affine_weight = nn.Parameter(torch.ones(num_features))
            self.affine_bias = nn.Parameter(torch.zeros(num_features))

    def forward(self, x, mode):
        # x: (B, L, C)
        if mode == "norm":
            self._get_statistics(x)
            x = self._normalize(x)
        elif mode == "denorm":
            x = self._denormalize(x)
        else:
            raise ValueError(f"mode '{mode}' không hợp lệ.")
        return x

    def _get_statistics(self, x):
        dim2reduce = tuple(range(1, x.ndim - 1))
        self.mean = torch.mean(x, dim=dim2reduce, keepdim=True).detach()
        self.stdev = torch.sqrt(
            torch.var(x, dim=dim2reduce, keepdim=True, unbiased=False) + self.eps
        ).detach()

    def _normalize(self, x):
        if self.non_norm:
            return x
        x = x - self.mean
        x = x / self.stdev
        if self.affine:
            x = x * self.affine_weight + self.affine_bias
        return x

    def _denormalize(self, x):
        if self.non_norm:
            return x
        if self.affine:
            x = (x - self.affine_bias) / (self.affine_weight + self.eps * self.eps)
        x = x * self.stdev
        x = x + self.mean
        return x


# =============================================================================
# 5. TIMEMIXER MODEL (forecasting)
# =============================================================================
class TimeMixer(nn.Module):
    """
    TimeMixer cho long/short-term forecasting.

    Input  : (B, seq_len, n_features)
    Output : (B, pred_len, target_dim)

    Tham số quan trọng:
        seq_len, pred_len : độ dài input / output
        n_features        : số feature input
        n_targets         : số target cần dự đoán
        target_indices    : list index của target trong feature_cols (cần khi
                            n_targets < n_features, model sẽ slice output)
        d_model, d_ff     : kích thước hidden
        e_layers          : số PDM block xếp chồng
        down_sampling_layers : số scale (ngoài scale gốc), tổng có
                               down_sampling_layers + 1 scale
        down_sampling_window : tỉ lệ downsample giữa các scale
        down_sampling_method : 'avg' | 'max' | 'conv'
        decomp_method     : 'moving_avg' | 'dft_decomp'
        moving_avg        : kernel size cho moving_avg decomp
        top_k             : số tần số top-k cho DFT decomp
        channel_independence : True = mỗi channel xử lý độc lập
        use_norm          : dùng instance normalization (RevIN-like)
    """

    def __init__(
        self,
        seq_len,
        pred_len,
        n_features,
        n_targets=None,
        target_indices=None,
        d_model=64,
        d_ff=128,
        e_layers=2,
        dropout=0.1,
        down_sampling_layers=2,
        down_sampling_window=2,
        down_sampling_method="avg",
        decomp_method="moving_avg",
        moving_avg=25,
        top_k=5,
        channel_independence=False,
        use_norm=True,
    ):
        super().__init__()

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.n_features = n_features
        self.n_targets = n_targets if n_targets is not None else n_features
        self.target_indices = target_indices
        self.down_sampling_layers = down_sampling_layers
        self.down_sampling_window = down_sampling_window
        self.down_sampling_method = down_sampling_method
        self.channel_independence = channel_independence
        self.use_norm = use_norm

        # ----- Embedding -----
        # channel_independence=True: mỗi channel xử lý như 1 chuỗi riêng
        embed_dim = 1 if channel_independence else n_features
        self.enc_embedding = nn.Sequential(
            nn.Linear(embed_dim, d_model),
            nn.Dropout(dropout),
        )

        # ----- PDM blocks -----
        self.pdm_blocks = nn.ModuleList([
            PastDecomposableMixing(
                seq_len=seq_len,
                d_model=d_model,
                d_ff=d_ff,
                dropout=dropout,
                down_sampling_window=down_sampling_window,
                down_sampling_layers=down_sampling_layers,
                decomp_method=decomp_method,
                moving_avg=moving_avg,
                top_k=top_k,
                channel_independence=channel_independence,
            )
            for _ in range(e_layers)
        ])

        # ----- Downsampling -----
        if down_sampling_method == "avg":
            self.down_pool = nn.AvgPool1d(kernel_size=down_sampling_window)
        elif down_sampling_method == "max":
            self.down_pool = nn.MaxPool1d(kernel_size=down_sampling_window)
        elif down_sampling_method == "conv":
            self.down_pool = nn.Conv1d(
                in_channels=n_features, out_channels=n_features,
                kernel_size=3, padding=1, stride=down_sampling_window,
                padding_mode="circular", bias=False,
            )
        else:
            raise ValueError(f"down_sampling_method '{down_sampling_method}' không hợp lệ.")

        # ----- Future Multipredictor Mixing (FMM) -----
        # Mỗi scale có 1 predictor riêng cho time dimension và 1 cho output channel
        self.predict_layers = nn.ModuleList([
            nn.Linear(
                seq_len // (down_sampling_window ** i),
                pred_len,
            )
            for i in range(down_sampling_layers + 1)
        ])

        # Project từ d_model về n_features
        if channel_independence:
            self.projection_layer = nn.Linear(d_model, 1, bias=True)
        else:
            self.projection_layer = nn.Linear(d_model, self.n_targets, bias=True)
            # Nếu channel_independence và muốn output đúng n_targets,
            # ta slice ở cuối bằng target_indices.

        # ----- Normalize -----
        if use_norm:
            # Mỗi scale có 1 normalize layer riêng
            self.normalize_layers = nn.ModuleList([
                Normalize(n_features, affine=True)
                for _ in range(down_sampling_layers + 1)
            ])

    # ---------- helper: downsample input thành multi-scale ----------
    def _multi_scale_process_inputs(self, x_enc):
        """
        x_enc: (B, L, C)
        Trả về list [(B, L, C), (B, L/k, C), (B, L/k^2, C), ...]
        """
        if self.down_sampling_method == "conv":
            x = x_enc.permute(0, 2, 1)  # (B, C, L)
        else:
            x = x_enc.permute(0, 2, 1)

        x_enc_list = [x_enc]
        x_curr = x
        for _ in range(self.down_sampling_layers):
            x_curr = self.down_pool(x_curr)
            x_enc_list.append(x_curr.permute(0, 2, 1))
        return x_enc_list

    # ---------- forward ----------
    def forward(self, x_enc):
        """
        x_enc: (B, seq_len, n_features)
        return: (B, pred_len, n_targets)
        """
        # 1. Multi-scale inputs
        x_list = self._multi_scale_process_inputs(x_enc)

        # 2. Normalize từng scale
        if self.use_norm:
            x_list = [self.normalize_layers[i](x, "norm") for i, x in enumerate(x_list)]

        # 3. Channel independence: gộp channel vào batch dimension
        if self.channel_independence:
            x_list = [
                x.permute(0, 2, 1).reshape(x.size(0) * x.size(2), x.size(1), 1)
                for x in x_list
            ]

        # 4. Embedding
        enc_out_list = [self.enc_embedding(x) for x in x_list]
        # enc_out_list: list các (B', L_i, d_model)

        # 5. Past Decomposable Mixing
        for pdm in self.pdm_blocks:
            enc_out_list = pdm(enc_out_list)

        # 6. Future Multipredictor Mixing
        dec_out_list = []
        for i, enc_out in enumerate(enc_out_list):
            # enc_out: (B', L_i, d_model)
            # predict theo trục thời gian: (B', d_model, L_i) → (B', d_model, pred_len)
            dec = self.predict_layers[i](enc_out.permute(0, 2, 1)).permute(0, 2, 1)
            # (B', pred_len, d_model) → (B', pred_len, out_dim)
            dec = self.projection_layer(dec)
            dec_out_list.append(dec)

        # Ensemble: sum các predictor
        dec_out = torch.stack(dec_out_list, dim=-1).sum(-1)
        # dec_out: (B', pred_len, out_dim)

        # 7. Channel independence: tách channel ra
        if self.channel_independence:
            B = x_enc.size(0)
            C = self.n_features
            dec_out = dec_out.reshape(B, C, self.pred_len, 1).squeeze(-1)
            dec_out = dec_out.permute(0, 2, 1)  # (B, pred_len, C)

        # 8. Denormalize (chỉ áp dụng cho scale gốc)
        if self.use_norm:
            if not self.channel_independence:
                # dec_out đang có out_dim = n_targets; cần map về n_features để denorm
                # Cách đơn giản: nếu n_targets < n_features, pad 0 các kênh không dự đoán
                if dec_out.size(-1) != self.n_features:
                    full = torch.zeros(
                        dec_out.size(0), dec_out.size(1), self.n_features,
                        device=dec_out.device, dtype=dec_out.dtype,
                    )
                    if self.target_indices is not None:
                        full[:, :, self.target_indices] = dec_out
                    else:
                        full[:, :, : dec_out.size(-1)] = dec_out
                    dec_out_full = full
                else:
                    dec_out_full = dec_out
                dec_out_full = self.normalize_layers[0](dec_out_full, "denorm")
                # Slice lại đúng target
                if self.target_indices is not None:
                    dec_out = dec_out_full[:, :, self.target_indices]
                else:
                    dec_out = dec_out_full[:, :, : self.n_targets]
            else:
                # channel_independence: denorm trực tiếp trên (B, pred_len, C)
                dec_out = self.normalize_layers[0](dec_out, "denorm")
                if self.target_indices is not None:
                    dec_out = dec_out[:, :, self.target_indices]
                else:
                    dec_out = dec_out[:, :, : self.n_targets]
        else:
            # Không normalize: nếu output đã đủ n_targets thì giữ nguyên
            if dec_out.size(-1) != self.n_targets:
                if self.target_indices is not None:
                    dec_out = dec_out[:, :, self.target_indices]
                else:
                    dec_out = dec_out[:, :, : self.n_targets]

        return dec_out

# Run

In [9]:
RANDOM_SEED = 42
SEED_WORKER = set_seed(RANDOM_SEED)
g = torch.Generator(); g.manual_seed(RANDOM_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Random seed: 42
Device: cuda


In [10]:
dataset = TimeSeriesDataset(
    name="beijing_air_quality",
    seq_len=24, pred_len=1,
    train_ratio=0.7, val_ratio=0.2, test_ratio=0.1,
    batch_size=64, normalize=True,
    worker_init_fn=SEED_WORKER, generator=g,
)
dataset.print_info()
train_loader, val_loader, test_loader = dataset.get_loaders()

=== DATASET: BEIJING_AIR_QUALITY ===
- Tổng số dòng: 35064
- seq_len / pred_len: 24 / 1 | batch_size: 64
- Normalize: True
- #features: 27 | #targets: 6
- X shape: (24, 27) | y shape: (1, 6)
- Split (train/val/test): 24544/7013/3507 dòng | 24520/6989/3483 cửa sổ


In [11]:
target_indices = dataset.get_target_indices_in_features()

model = TimeMixer(
    seq_len=dataset.seq_len,
    pred_len=dataset.pred_len,
    n_features=len(dataset.feature_cols),
    n_targets=len(dataset.target_cols),
    target_indices=target_indices,
    d_model=64,
    d_ff=128,
    e_layers=2,
    dropout=0.1,
    down_sampling_layers=2,        # tổng 3 scale (gốc + 2 downsample)
    down_sampling_window=2,
    down_sampling_method="avg",    # 'avg' | 'max' | 'conv'
    decomp_method="moving_avg",    # 'moving_avg' | 'dft_decomp'
    moving_avg=25,
    top_k=5,
    channel_independence=False,
    use_norm=True,                 # RevIN-like instance norm
)

In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

trainer = Trainer(
    model=model, dataset=dataset,
    train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
    criterion=nn.MSELoss(),
    optimizer=optimizer, scheduler=scheduler,
    device=DEVICE,
)
trainer.fit(num_epochs=80, patience=15, eval_every=5)

Epoch [001/80] Train: 0.178194 | Val: 0.086355 | lr: 1.00e-03
Epoch [002/80] Train: 0.103281 | Val: 0.075415 | lr: 1.00e-03
Epoch [003/80] Train: 0.096095 | Val: 0.074475 | lr: 1.00e-03
Epoch [004/80] Train: 0.092768 | Val: 0.068533 | lr: 1.00e-03
Epoch [005/80] Train: 0.091451 | Val: 0.068923 | lr: 1.00e-03 | Val MAE: 33.8884
Epoch [006/80] Train: 0.090607 | Val: 0.068261 | lr: 1.00e-03
Epoch [007/80] Train: 0.088060 | Val: 0.066810 | lr: 1.00e-03
Epoch [008/80] Train: 0.087577 | Val: 0.067552 | lr: 1.00e-03
Epoch [009/80] Train: 0.086163 | Val: 0.064739 | lr: 1.00e-03
Epoch [010/80] Train: 0.085876 | Val: 0.065223 | lr: 1.00e-03 | Val MAE: 32.0851
Epoch [011/80] Train: 0.085547 | Val: 0.065644 | lr: 1.00e-03
Epoch [012/80] Train: 0.084322 | Val: 0.065416 | lr: 1.00e-03
Epoch [013/80] Train: 0.083300 | Val: 0.065550 | lr: 5.00e-04
Epoch [014/80] Train: 0.080268 | Val: 0.063009 | lr: 5.00e-04
Epoch [015/80] Train: 0.079128 | Val: 0.063028 | lr: 5.00e-04 | Val MAE: 31.4038
Epoch [016/80

{'train_loss': [0.17819382813295767,
  0.10328127463480388,
  0.09609543224629531,
  0.0927679309132243,
  0.09145075631909814,
  0.0906074905424678,
  0.08806007719594153,
  0.08757683442720192,
  0.08616346057510493,
  0.08587621758657023,
  0.08554726505551798,
  0.08432214749190002,
  0.08329951305179378,
  0.08026847212923292,
  0.07912764461501387,
  0.07823075933129714,
  0.07832101613137306,
  0.07719616858838628,
  0.0751761496601836,
  0.0741649604786862,
  0.07393196628202622,
  0.07370652294980566,
  0.07335722433679645,
  0.07297475276116835,
  0.07239471027296383,
  0.07123605735223211,
  0.07060548755268872,
  0.07042040548623095,
  0.06991524169036167,
  0.06947732654568811,
  0.06902619100997733,
  0.06953434998604058,
  0.06872458765389577,
  0.06849055048108685,
  0.0686597328468987,
  0.06829388410971177],
 'val_loss': [0.08635450917590473,
  0.07541498985412824,
  0.07447545036608517,
  0.06853337439531582,
  0.06892321885030811,
  0.06826097721686766,
  0.06681038

In [13]:
baseline_metrics = evaluate_naive_baseline(dataset, test_loader)
model_metrics = trainer.evaluate(test_loader, title="MODEL TEST")
compare_metrics(baseline_metrics, model_metrics)

=== NAIVE BASELINE ===
     PM2.5 | MSE: 560.0901 | MAE: 12.4416 | RMSE: 23.6662 | WMAPE: 0.1228 | R2: 0.9499
      PM10 | MSE: 857.7209 | MAE: 16.7040 | RMSE: 29.2869 | WMAPE: 0.1425 | R2: 0.9291
       SO2 | MSE: 28.3353 | MAE: 2.7752 | RMSE: 5.3231 | WMAPE: 0.1858 | R2: 0.8969
       NO2 | MSE: 231.6196 | MAE: 8.4844 | RMSE: 15.2191 | WMAPE: 0.1182 | R2: 0.8724
        CO | MSE: 370941.7188 | MAE: 288.5731 | RMSE: 609.0498 | WMAPE: 0.1591 | R2: 0.8776
        O3 | MSE: 135.7468 | MAE: 6.3250 | RMSE: 11.6510 | WMAPE: 0.2106 | R2: 0.8545
--------------------------------------------------------------------------------
OVERALL    | MSE: 62125.8750 | MAE: 55.8839 | RMSE: 249.2506 | WMAPE: 0.1560 | R2: 0.9335
=== MODEL TEST ===
     PM2.5 | MSE: 476.2021 | MAE: 12.2333 | RMSE: 21.8221 | WMAPE: 0.1208 | R2: 0.9574
      PM10 | MSE: 883.5596 | MAE: 17.4987 | RMSE: 29.7247 | WMAPE: 0.1492 | R2: 0.9270
       SO2 | MSE: 26.6470 | MAE: 2.7894 | RMSE: 5.1621 | WMAPE: 0.1868 | R2: 0.9031
       